In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from tqdm import tqdm
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    pseudogene_data = []
    with tqdm(total = len(org_data_n['accession']), desc=f'{genus_name}({len(org_data_n['accession'])})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for i in range(len(org_data_n['accession'])):
            acc_n = org_data_n['accession'][i]
            pbar.update(1)
            handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
            seq_record = SeqIO.parse(handle, 'genbank')
            for record in seq_record:
                CDS_count = {'contig': f'{acc_n}-{record.id}', 'CDSs (total)': 0, 'CDSs (with protein)': 0, 'CDSs (without protein, Pseudo Genes)': 0}
                for feature in record.features:
                    if feature.type == 'CDS':
                        CDS_count['CDSs (total)'] += 1
                        if 'translation' in feature.qualifiers:
                            CDS_count['CDSs (with protein)'] += 1
                        else:
                            CDS_count['CDSs (without protein, Pseudo Genes)'] += 1
                pseudogene_data.append(pd.DataFrame([CDS_count]))

    
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    os.chdir(folder)
    pseudogene_pd = pd.concat(pseudogene_data, ignore_index=True)
    pseudogene_pd.to_csv('pseudogene_statistics.tsv', sep='\t', index=False)

Escherichia(4204): 100%|████████████████████████████████████████| 4.20k/4.20k [39:56<00:00, 1.75B/s]
Klebsiella(3554): 100%|█████████████████████████████████████████| 3.55k/3.55k [37:06<00:00, 1.60B/s]
Staphylococcus(2423): 100%|█████████████████████████████████████| 2.42k/2.42k [11:05<00:00, 3.64B/s]
Pseudomonas(2343): 100%|████████████████████████████████████████| 2.34k/2.34k [24:34<00:00, 1.59B/s]
Bacillus(1976): 100%|███████████████████████████████████████████| 1.98k/1.98k [15:15<00:00, 2.16B/s]
Salmonella(1853): 100%|█████████████████████████████████████████| 1.85k/1.85k [15:10<00:00, 2.04B/s]
Streptococcus(1599): 100%|██████████████████████████████████████| 1.60k/1.60k [05:11<00:00, 5.14B/s]
Streptomyces(1359): 100%|███████████████████████████████████████| 1.36k/1.36k [17:56<00:00, 1.26B/s]
Acinetobacter(1234): 100%|██████████████████████████████████████| 1.23k/1.23k [07:30<00:00, 2.74B/s]
Helicobacter(416): 100%|████████████████████████████████████████████| 416/416 [01:00<00:00,